# Multi-Geometry Stitching

Combine angle-separated detector images with one calibrated integrator
per image. Normalization is a per-image monitor factor, not a visual
adjustment; inspect overlap and correction assumptions before analysis.


In [ ]:
import os
from IPython import get_ipython

# Equivalent to %matplotlib widget; headless checks explicitly use inline.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", os.environ.get("XDART_NOTEBOOK_BACKEND", "widget"))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.analysis import StitchPlan, run_stitch
from xrd_tools.core.containers import PONI
from xrd_tools.core.scan import ScanFrame
from xrd_tools.integrate import load_poni
from xrd_tools.io import load_mask, read_image
from xrd_tools.sources import MemoryFrameSource
from xrd_tools.viz import plot_1d


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
image_paths = []  # Real mode: naturally ordered detector images.
poni_file = None  # Real mode: persisted base PONI.
rotation_values = []  # Real mode: rot1 degrees, one per image.
monitor_values = []   # Real mode: optional monitor, one per image.
mask_file = None      # Real mode: optional detector mask.
q_range = widgets.FloatRangeSlider(value=(1.0, 4.0), min=0.5, max=6.0, step=0.05, description="q range", continuous_update=False)
monitor_key = widgets.Text(value="i0", description="monitor")
use_mask = widgets.Checkbox(value=False, description="apply mask")
compute = widgets.Button(description="Compute stitch", button_style="primary")
status = widgets.HTML("<i>Compute performs both 1-D and 2-D StitchPlan runs.</i>")
output = widgets.Output()
display(widgets.VBox([q_range, monitor_key, use_mask, compute, status, output]))


In [ ]:
NOTEBOOK_STATE = {"runs": 0, "stitch_1d": None, "stitch_2d": None}

def _smoke_stitch_source():
    shape = (195, 487)
    poni = PONI(dist=0.2, poni1=shape[0] * 172e-6 / 2, poni2=shape[1] * 172e-6 / 2,
                rot1=0.0, rot2=0.0, rot3=0.0, wavelength=1e-10, detector="Pilatus100k")
    yy, xx = np.mgrid[:shape[0], :shape[1]]
    radius = np.sqrt((yy - shape[0] / 2) ** 2 + (xx - shape[1] / 2) ** 2)
    base = 300 * np.exp(-((radius - 55) / 9) ** 2) + 2
    frames = [ScanFrame(index, image=base * (1 + 0.03 * index), metadata={"rot1": 4.0 * index, "i0": 1.0 + 0.02 * index}) for index in range(3)]
    return MemoryFrameSource(frames, name="synthetic_stitch"), poni

def _real_stitch_source():
    assert image_paths, "Configure image_paths for real stitching"
    assert poni_file is not None, "Configure a persisted base poni_file"
    paths = sorted(map(Path, image_paths), key=lambda path: path.name)
    assert len(rotation_values) == len(paths), "rotation_values must match image_paths"
    if monitor_values:
        assert len(monitor_values) == len(paths), "monitor_values must match image_paths"
    mask = load_mask(mask_file) if use_mask.value and mask_file else None
    frames = [ScanFrame(index, image=read_image(path, mask=mask), metadata={"rot1": float(rotation_values[index]), "i0": float(monitor_values[index]) if monitor_values else 1.0}) for index, path in enumerate(paths)]
    return MemoryFrameSource(frames, name="real_stitch"), load_poni(poni_file)

def compute_stitch(_=None):
    with output:
        clear_output(wait=True)
        try:
            source, poni = _smoke_stitch_source() if SMOKE_MODE else _real_stitch_source()
            common = dict(base_poni=poni, rot1_key="rot1", monitor_key=monitor_key.value or None,
                          radial_range=tuple(q_range.value), npt_1d=180, npt_rad_2d=120, npt_azim_2d=48)
            one_d = run_stitch(StitchPlan(mode="1d", **common), source).payload
            two_d = run_stitch(StitchPlan(mode="2d", **common), source).payload
            fig, axes = plt.subplots(1, 2, figsize=(11, 3))
            plot_1d(axes[0], one_d.radial, one_d.intensity, fmt="-", attrs={"xlabel": one_d.unit, "ylabel": "Intensity", "title": "StitchPlan 1-D"})
            axes[1].pcolormesh(two_d.radial, two_d.azimuthal, two_d.intensity.T, shading="auto")
            axes[1].set(xlabel=two_d.unit, ylabel=two_d.azimuthal_unit, title="StitchPlan 2-D")
            plt.show()
            NOTEBOOK_STATE.update(runs=NOTEBOOK_STATE["runs"] + 1, stitch_1d=one_d, stitch_2d=two_d)
            status.value = f"<b>Stitched {len(source.frame_indices)} images in 1-D and 2-D.</b>"
        except Exception as exc:
            status.value = f"<b>Stitch failed:</b> {exc}"
            raise

compute.on_click(compute_stitch)
NOTEBOOK_ACTIONS = {"compute_stitch": compute_stitch}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    compute_stitch()
